# Historical FER experiment, full notebook unrerun
No online setup is executed by the package. Training/validation/final test are distinct Usage partitions. Emotion labels are dataset categories, not reliable measurements of a person's internal feelings.


In [ ]:
# Historical online setup disabled. Use explicit local CSV with Usage splits.


In [ ]:
# Historical online setup disabled. Use explicit local CSV with Usage splits.


In [ ]:
# Historical online setup disabled. Use explicit local CSV with Usage splits.


In [ ]:
# Historical online setup disabled. Use explicit local CSV with Usage splits.


In [ ]:
# Historical online setup disabled. Use explicit local CSV with Usage splits.


In [ ]:
import pandas as pd

# Load the dataset
data = pd.read_csv('fer2013.csv')


In [ ]:
data.head()

In [ ]:
import numpy as np

# Split data into pixels and labels
pixels = data['pixels'].tolist()
labels = data['emotion'].tolist()

# Convert pixels to numpy arrays and reshape into 48x48 images
images = np.array([np.fromstring(p, dtype=int, sep=' ').reshape(48, 48) for p in pixels])
images = images / 255.0  # Normalize pixel values to [0, 1]
labels = np.array(labels)


In [ ]:
# Respect the provided Usage partition. Do not use final test as validation.
X_train, y_train = images[data.Usage == "Training"], labels[data.Usage == "Training"]
X_val, y_val = images[data.Usage == "PublicTest"], labels[data.Usage == "PublicTest"]
X_test, y_test = images[data.Usage == "PrivateTest"], labels[data.Usage == "PrivateTest"]


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

model = models.Sequential()
model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(48, 48, 1)))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation='relu'))
model.add(layers.Flatten())
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dense(7, activation='softmax'))  # 7 classes for emotions

model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])
model.summary()

In [ ]:
X_train = np.expand_dims(X_train, -1)
X_val = np.expand_dims(X_val, -1)
X_test = np.expand_dims(X_test, -1)
history = model.fit(X_train, y_train, epochs=10, validation_data=(X_val, y_val))


In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print("Test Accuracy:", test_acc)


In [ ]:
model.save("facial_expression_model.keras")


In [ ]:
model = tf.keras.models.load_model("facial_expression_model.keras", safe_mode=True)


In [ ]:
import numpy as np

# Example: Using the first image from the test set
input_image = X_test[0]  # Select an image
input_image = np.expand_dims(input_image, axis=0)  # Add batch dimension


In [ ]:
prediction = model.predict(input_image)
predicted_class = np.argmax(prediction)  # Get the class with the highest probability
print("Predicted Emotion Class:", predicted_class)


In [ ]:
emotions = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']
print("Predicted Emotion:", emotions[predicted_class])
